In [1]:
# Load environment and initialize the model
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpointEmbeddings

load_dotenv(override=True)

# Initialize Groq model
llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0.7
)

# Load embeddings
hf_token = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=hf_token,
)

# Load vectorstore from disk - no need to rebuild!
vectorstore = FAISS.load_local(
    "../data/vectorstore",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Model loaded!")
print("✅ Vectorstore loaded!")
print("✅ Ready to build agent!")

c:\Users\Raheem Khan\Desktop\countryside-resort-bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Model loaded!
✅ Vectorstore loaded!
✅ Ready to build agent!


In [2]:
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# Define duck_search first!
duck_search = DuckDuckGoSearchRun()

# --- Tool 1: RAG Search ---
@tool
def search_resort_info(query: str) -> str:
    """Search Countryside Resort documents for information about
    rooms, prices, activities, policies, check-in time, amenities,
    nearby places to visit in Gilgit, local food and attractions.
    Use this as the FIRST tool for any question."""
    
    results = retriever.invoke(query)
    combined = "\n\n".join([doc.page_content for doc in results])
    return combined

# --- Tool 2: Web Search ---
@tool
def search_web(query: str) -> str:
    """Search the web for additional live information about tourist 
    attractions, places to visit, travel tips, road conditions, 
    weather and activities in Gilgit city and Gilgit-Baltistan, Pakistan.
    Use this AFTER search_resort_info for extra current information."""
    
    try:
        gilgit_query = f"{query} Gilgit Gilgit-Baltistan Pakistan"
        return duck_search.run(gilgit_query)
    except Exception:
        return "Web search temporarily unavailable."

# --- Tool 3: Contact Info ---
@tool
def get_contact_info(query: str) -> str:
    """Returns Countryside Resort contact information including
    phone number, email, location and social media.
    Use this when guest asks for phone, email, address or how to book."""
    
    return """
    Countryside Resort Gilgit - Contact Information:
    📞 Phone: 03109152096 / 03555033760
    📧 Email: Countrysideresortglt@gmail.com
    📍 Location: Jageer Baseen, Gilgit-Baltistan (6km from airport)
    🌐 Website: https://www.countrysideresortgilgit.com
    📱 Social: @Countrysideresortgilgit (Facebook & Instagram)
    🏦 Bank: Habib Metropolitan Bank
    🔢 IBAN: PK61 MPBL 0286 0271 4013 9401
    """

tools = [search_resort_info, search_web, get_contact_info]
print(f"✅ {len(tools)} tools created!")
for t in tools:
    print(f"  - {t.name}: {t.description[:70]}...")

✅ 3 tools created!
  - search_resort_info: Search Countryside Resort documents for information about
    rooms, p...
  - search_web: Search the web for additional live information about tourist 
    attr...
  - get_contact_info: Returns Countryside Resort contact information including
    phone num...


In [3]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    prompt="""You are Ali, a helpful assistant for Countryside Resort Gilgit,
    located in the beautiful Gilgit-Baltistan region of Pakistan.
    
    IMPORTANT RULES:
    - ALWAYS call a tool before answering. Never answer from your own knowledge.
    - For resort questions (rooms, prices, policies, amenities) → use search_resort_info tool
    - For places to visit and local attractions → use search_resort_info tool FIRST,
      then also use search_web tool to add any extra current information
    - For contact details → use get_contact_info tool
    - After getting tool results, give a detailed, helpful and complete answer
    - Never give a one line answer - always be thorough
    - List specific places like Kargah Buddha, Naltar Valley, Kargha Nala with descriptions
    
    Always be warm, friendly and professional.
    Always respond in English.
    """
)

print("✅ Agent ready!")

✅ Agent ready!


C:\Users\Raheem Khan\AppData\Local\Temp\ipykernel_1784\4281240779.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [4]:
def chat(question, thread_id="guest-1"):
    print(f"\n🧳 Guest: {question}")
    print("-" * 50)
    
    config = {"configurable": {"thread_id": thread_id}}
    
    response = agent_executor.invoke(
        {"messages": [("human", question)]},
        config=config
    )
    
    answer = response["messages"][-1].content
    print(f"🤖 Ali: {answer}")
    print("-" * 50)

In [5]:
chat("What are the best places to visit near the resort?", thread_id="guest-3")


🧳 Guest: What are the best places to visit near the resort?
--------------------------------------------------
🤖 Ali: Based on the search results, the top places to visit near the resort are:

1. Kargah Buddha - An ancient 7th-century rock carving of Buddha, located just 10 minutes from the city.
2. Naltar Valley - Famous for its colorful lakes and skiing opportunities, it's about 40 km from Gilgit.
3. Rakaposhi Viewpoint - Offers stunning views of the Rakaposhi (7788m) peak.
4. Attabad Lake - A stunning turquoise lake in Hunza, located about 2 hours from Gilgit.
5. Baltit Fort - A historic fort in Karimabad Hunza, about 2 hours away.

Additionally, the search results suggest that Gilgit-Baltistan is a region with a lot to offer, including lush valleys, pristine lakes, and glacial waterfalls. It's a great destination for nature lovers and those looking for adventure.

It's also worth noting that the best time to visit Gilgit is during the spring (April-May), summer (June-August), autu

In [6]:
# Test 1 - room price
chat("What is the price of Standard Room?", thread_id="guest-1")




🧳 Guest: What is the price of Standard Room?
--------------------------------------------------
🤖 Ali: Based on the information provided by the resort, the price of a Standard Room varies depending on the season. Here are the rates for a Standard Room:

* Low season: 6,000 PKR per night
* Base season: 7,500 PKR per night
* Peak season: 9,000 PKR per night
* Eid/Holidays: 11,000 PKR per night

Please note that these rates are subject to change and may be subject to additional taxes or fees. It's always best to check with the resort directly for the most up-to-date and accurate pricing information.
--------------------------------------------------


In [7]:
# Test 2 - does it remember we asked about Standard Room?
chat("And what does that room include?", thread_id="guest-1")




🧳 Guest: And what does that room include?
--------------------------------------------------
🤖 Ali: Based on the information provided by the resort, a Standard Room includes the following:

* Complimentary breakfast for 2 adults
* Free WiFi
* Pool access
* Orchard access
* Free parking

These amenities are included with every room type, regardless of the season or rate.
--------------------------------------------------


In [8]:
# Test 3 - does it remember the whole conversation?
chat("Can I bring my family of 4 to that room?", thread_id="guest-1")




🧳 Guest: Can I bring my family of 4 to that room?
--------------------------------------------------
🤖 Ali: Based on the information provided by the resort, the Standard Room has a capacity of 2 adults. However, the resort allows families with 2 adults and 2 kids below 12 years old per room, except for the Standard Room, which has a capacity of 2 beds only.

So, in your case, you can bring your family of 4 to the Deluxe / Queen Room or the Standard Large / Twin room, but not to the Standard Room.
--------------------------------------------------


In [9]:
# Test 4 - contact info
chat("How can I book it? Give me the contact details", thread_id="guest-1")


🧳 Guest: How can I book it? Give me the contact details
--------------------------------------------------
🤖 Ali: To book a room at Countryside Resort Gilgit, you can contact them through the following details:

Phone: 03109152096 / 03555033760
Email: Countrysideresortglt@gmail.com
Location: Jageer Baseen, Gilgit-Baltistan (6km from airport)
Website: https://www.countrysideresortgilgit.com
Social media: @Countrysideresortgilgit (Facebook & Instagram)

You can also deposit the booking amount in the following bank account:

Bank: Habib Metropolitan Bank
IBAN: PK61 MPBL 0286 0271 4013 9401
--------------------------------------------------
